In [ ]:
# Function to generate random timestamp
def random_timestamp(start, end):
    delta = end - start
    int_delta = delta.days * 24 * 60 * 60 + delta.seconds
    random_second = random.randrange(int_delta)
    return start + timedelta(seconds=random_second)

# Start and end date for random timestamp generation
start_date = datetime(2021, 6, 3)
end_date = datetime.now()

# Generate 50 records
records = []
for _ in range(50):
    record = (
        f"CUST_{random.randint(1000, 9999)}",  # Random CUSTOMER
        f"Z{random.randint(1, 9)}",  # Random CUSTOMER_CODE
        random.choice(["US", "CA"]),  # Random COUNTRY
        "EN",  # LANGUAGE
        random.randint(10, 24),  # Random CREATEDBY
        random_timestamp(start_date, end_date),  # Random CREATEDAT
        random.randint(10, 24),  # Random CHANGEDBY
        random_timestamp(start_date, end_date)
    )
    records.append(record)

schema = StructType([
    StructField("CUSTOMERID", StringType(), True),
    StructField("CUSTOMERTYPEID", StringType(), True),
    StructField("COUNTRY", StringType(), True),
    StructField("LANGUAGE", StringType(), True),
    StructField("CREATEDBY", IntegerType(), True),
    StructField("CREATEDAT", TimestampType(), True),
    StructField("CHANGEDBY", IntegerType(), True),
    StructField("CHANGEDAT", TimestampType(), True)
])

# Create DataFrame
df_cust = spark.createDataFrame(records, schema)

# Convert date columns to timestamp
df_cust = df_cust.withColumn("CREATEDAT", col("CREATEDAT").cast(TimestampType())) \
       .withColumn("CHANGEDAT", col("CHANGEDAT").cast(TimestampType()))

# Show the DataFrame
# df_cust.display()


In [ ]:

from pyspark.sql.functions import col
from pyspark.sql.types import StringType, IntegerType, TimestampType
import random
from datetime import datetime, timedelta



# Function to generate random timestamp
def random_timestamp(start, end):
    delta = end - start
    int_delta = delta.days * 24 * 60 * 60 + delta.seconds
    random_second = random.randrange(int_delta)
    return start + timedelta(seconds=random_second)

# Start and end date for random timestamp generation
start_date = datetime(2018, 1, 1)
end_date = datetime.now()

# Generate 50 records
records = []
for i in range(50):
    record = (
        random.randint(100000, 999999),  # Random TRANSACTIONID
        f"FPA1/{random.randint(1, 99):03}",  # Random ACCOUNTID
        "EXP",  # ACCOUNT
        # f"CUST_{random.randint(1000, 9999)}",  # Random CUSTOMERID
        df_cust.select('CUSTOMERID').collect()[i]['CUSTOMERID'],
        f"Z{random.randint(1, 9)}",  # Random CUSTOMER_CODE
        f"PRD_{random.randint(100, 999)}",  # Random PRODUCTID
        random.choice(["MT", "FG", "PK"]),  # Random PRODUCTCAT
        "US10_PCM",  # PROFITCENTER
        random.choice(["Budget", "Actual"]),  # Random VERSION
        random_timestamp(start_date, end_date),  # Random DATE
        random.randint(1000, 100000)  # Random VALUE
    )
    records.append(record)

schema1 = StructType([
    StructField("TRANSACTIONID", IntegerType(), True),
    StructField("ACCOUNTID", StringType(), True),
    StructField("ACCOUNTTYPEID", StringType(), True),
    StructField("CUSTOMERID", StringType(), True),
    StructField("CUSTOMERTYPEID", StringType(), True),
    StructField("PRODUCTID", StringType(), True),
    StructField("PRODUCTCATEGORYID", StringType(), True),
    StructField("PROFITCENTERID", StringType(), True),
    StructField("VERSION", StringType(), True),
    StructField("DATE", TimestampType(), True),
    StructField("VALUE", IntegerType(), True)
])

# Create DataFrame
df_finance = spark.createDataFrame(records, schema1)

# Convert DATE column to timestamp
df_finance = df_finance.withColumn("DATE", col("DATE").cast(TimestampType()))

# Show the DataFrame
# df_finance.display()


In [ ]:

from pyspark.sql.functions import *
file_path = "/FileStore/tables/Customer.csv"
df1 = spark.read.csv(file_path, header=True, inferSchema=True)
df1 = df1.withColumn("CREATEDAT", col("CREATEDAT").cast(TimestampType())) \
       .withColumn("CHANGEDAT", col("CHANGEDAT").cast(TimestampType()))

cust_df = df1.union(df_cust)
cust_df.createOrReplaceTempView('cust_tbl')

In [ ]:
file_path = "/FileStore/tables/FinancialTransactions.csv"
df2= spark.read.csv(file_path, header=True, inferSchema=True)
df2 = df2.withColumn("DATE", col("DATE").cast(TimestampType()))

finance_df = df2.union(df_finance)
# finance_df.show()
finance_df.createOrReplaceTempView('finance_tbl')